<h1 style="background-color:#B89235; color:white;">Práctica 3: Redes neuronales LSTM </h1>

**Alumna:** Erika Margarita Villaobos Martínez

**Fecha de entrega:** 9 de abril 2026

___

<h2 style="background-color:#DEE4FF; color:#B89235;">Paso 0: Librerias y lectura de archivos </h2> 

<h2 style="color:#2749F5; font-size: 1.2em; font-weight: bold;"> Librerias </h2>

In [1]:
## Generales
import pandas as pd
import numpy as np
import glob
import os

## Gráficos
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.express as px

<h2 style="color:#2749F5; font-size: 1.2em; font-weight: bold;"> Lectura de archivo </h2>

In [2]:
ruta = "../data/practica_3/historicos"

In [3]:
archivos = glob.glob(os.path.join(ruta, "meteorología_*.json"))

In [4]:
dfs = []

In [5]:
%%time
for archivo in archivos:
    with open(archivo, "r", encoding="utf-8") as f:
        data = json.load(f)

    dates = data["pollutionMeasurements"]["date"]

    rows = []
    for fecha, variables in dates.items():
        # Manejar el caso de "24:00"
        if "24:00" in fecha:
            base, hora = fecha.split(" ")
            fecha = pd.to_datetime(base) + pd.Timedelta(days=1)
            fecha = fecha.strftime("%Y-%m-%d 00:00")

        # Solo tomar TMP
        if "TMP" in variables:
            estaciones = variables["TMP"]
            if "FAC" in estaciones:  # Solo estación FES Acatlán
                valor = estaciones["FAC"]
                # Manejar valores vacíos
                if valor == "" or valor is None:
                    val = None
                else:
                    try:
                        val = float(valor)
                    except ValueError:
                        val = None

                rows.append({
                    "fecha": pd.to_datetime(fecha),
                    "TMP_FAC": val
                })

    df = pd.DataFrame(rows)
    dfs.append(df)

CPU times: total: 37.1 s
Wall time: 37.2 s


In [6]:
df_total = pd.concat(dfs, ignore_index=True)

In [7]:
df_total.head()

,fecha,TMP_FAC
0,2013-01-01 01:00:00,NaN
1,2013-01-01 02:00:00,NaN
2,2013-01-01 03:00:00,NaN
3,2013-01-01 04:00:00,NaN
4,2013-01-01 05:00:00,NaN


<h2 style="color:#2749F5; font-size: 1.2em; font-weight: bold;"> Calidad de datos </h2>

In [ ]:
# Asegúrate de que la columna fecha sea datetime
df_total["fecha"] = pd.to_datetime(df_total["fecha"])

In [ ]:
# Crear columnas de año y mes
df_total["año"] = df_total["fecha"].dt.year
df_total["mes"] = df_total["fecha"].dt.month

In [ ]:
# Agrupar por año y mes y calcular promedio
#df_mensual = df_total.groupby(["año", "mes"], as_index=False)["TMP_FAC"].mean()
df_estacional = df_total.groupby(df_total["fecha"].dt.month)["TMP_FAC"]

In [ ]:
# Crear columna de fecha representativa (primer día del mes)
df_mensual["fecha"] = pd.to_datetime(df_mensual["año"].astype(str) + "-" + df_mensual["mes"].astype(str) + "-01")

In [ ]:
# Graficar con Plotly
fig = px.line(df_mensual, x="fecha", y="TMP_FAC",
              title="Promedio mensual de temperatura en FES Acatlán (FAC)",
              labels={"TMP_FAC":"Temperatura promedio (°C)", "fecha":"Fecha"})
fig.show()

<h2 style="background-color:#DEE4FF; color:#B89235;">Paso 1: Análisis y preprocesamiento de datos </h2> 

<h2 style="color:#2749F5; font-size: 1.2em; font-weight: bold;"> EDA </h2>

In [ ]:
df_total.isna().sum()

In [ ]:
df.info()

In [ ]:
# --- Imputación mensual ---
%%time
df_total["TMP_FAC"] = df_total.groupby([df_total.index.year, df_total.index.month])["TMP_FAC"].transform(
    lambda x: x.fillna(x.mean())
)

In [ ]:
# --- Visualización de tendencia y estacionalidad ---
df_mensual = df_total.resample("M").mean()
plt.figure(figsize=(12,6))
plt.plot(df_mensual.index, df_mensual["TMP_FAC"], color="orange")
plt.title("Temperatura mensual promedio FAC (2013–2023)")
plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Normalización
scaler = MinMaxScaler()
df_total["TMP_scaled"] = scaler.fit_transform(df_total[["TMP_FAC"]])

In [ ]:
# Construcción de variable objetivo: temperatura t+n (ejemplo: 24 horas adelante)
n_steps = 24
X, y = [], []
values = df_total["TMP_scaled"].values
for i in range(len(values)-n_steps):
    X.append(values[i:i+n_steps])
    y.append(values[i+n_steps])
X, y = np.array(X), np.array(y)

# Reshape para LSTM [samples, timesteps, features]
X = X.reshape((X.shape[0], X.shape[1], 1))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
# Modelo LSTM
model = Sequential([
    LSTM(50, activation="tanh", input_shape=(n_steps, 1)),
    Dense(1)
])

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")

In [ ]:
# Entrenamiento
history = model.fit(X, y, epochs=20, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
# Visualizar pérdida
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.legend()
plt.title("Evolución de la pérdida")
plt.show()

In [ ]:
# Predicciones sobre el conjunto
y_pred = model.predict(X)

In [ ]:
# Invertir normalización
y_pred_inv = scaler.inverse_transform(y_pred)
y_true_inv = scaler.inverse_transform(y.reshape(-1,1))

In [ ]:
# Comparación gráfica
plt.figure(figsize=(12,6))
plt.plot(y_true_inv[:500], label="Real", color="blue")
plt.plot(y_pred_inv[:500], label="Predicho", color="red", alpha=0.7)
plt.title("Predicción de temperatura FAC (primeros 500 pasos)")
plt.xlabel("Tiempo")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.show()

In [ ]:
# Estimar promedio diario del mes siguiente
# Tomamos últimos datos y generamos predicciones hacia adelante
last_seq = values[-n_steps:]
preds = []

In [ ]:
for _ in range(30*24):  # 30 días * 24 horas
    x_input = last_seq.reshape((1, n_steps, 1))
    yhat = model.predict(x_input, verbose=0)
    preds.append(yhat[0,0])
    last_seq = np.append(last_seq[1:], yhat[0,0])

In [ ]:
preds_inv = scaler.inverse_transform(np.array(preds).reshape(-1,1))
promedio_diario = preds_inv.reshape(30,24).mean(axis=1)

In [ ]:
print("Temperatura promedio estimada por día del próximo mes:")
print(promedio_diario)

In [ ]:
# Asegúrate de que la columna fecha sea datetime y esté como índice
df_total["fecha"] = pd.to_datetime(df_total["fecha"])
df_total = df_total.set_index("fecha")

In [ ]:
# Filtrar mayo 2023
df_mayo = df_total.loc["2023-05-01":"2023-05-31"]

In [ ]:
# Agrupar por día y calcular promedio
df_mayo_diario = df_mayo.resample("D").mean()

In [ ]:
print(df_mayo_diario.head(10))  # primeros 10 días

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df_mayo_diario.index, df_mayo_diario["TMP_FAC"], marker="o", color="orange")
plt.title("Temperatura promedio diaria en FAC - Mayo 2023")
plt.xlabel("Día")
plt.ylabel("Temperatura (°C)")
plt.grid(True)
plt.show()